# Async Python

## asyncio Fundamentals

### The Event Loop

The event loop is the scheduler for async code.

It runs a coroutine until that coroutine reaches an await. If the coroutine is waiting on I/O, such as an HTTP request, the event loop can run another coroutine in the meantime. When the I/O is ready, the original coroutine continues.

Most application code does not manage the event loop directly. You write async functions and use asyncio.run() at the edge of the program.

### async def and await

async def defines a coroutine function. Calling it creates a coroutine object; the body does not run until the coroutine is awaited or scheduled.

await pauses the current coroutine until the awaited operation finishes. While it is paused, the event loop can let other coroutines run.

In [10]:
import asyncio
import time

async def call_llm(prompt: str, delay: float) -> str:
    """Pretend to call an LLM API that takes `delay` seconds."""
    print(f"Sending: {prompt}")
    await asyncio.sleep(delay)
    print(f"Received response for: {prompt}")
    return f"Response to: {prompt}"

async def main():
    start = time.perf_counter()

    result1 = await call_llm("What is Python?", 0.2)
    result2 = await call_llm("What is async?", 0.3)
    result3 = await call_llm("What is an LLM?", 0.1)

    elapsed = time.perf_counter() - start
    print(result1)
    print(result2)
    print(result3)
    print(f"Sequential: {elapsed:.1f} seconds")  # ~0.6 seconds

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())


Sending: What is Python?
Received response for: What is Python?
Sending: What is async?
Received response for: What is async?
Sending: What is an LLM?
Received response for: What is an LLM?
Response to: What is Python?
Response to: What is async?
Response to: What is an LLM?
Sequential: 0.6 seconds


This is a common point of confusion: async syntax alone does not create concurrency. To overlap the calls, you must schedule them together. We will do that with asyncio.gather()

One important detail: await asyncio.sleep(delay) is non-blocking. It gives control back to the event loop. By contrast, time.sleep(delay) blocks the whole thread. Do not use time.sleep() inside async code

### asyncio.run()

asyncio.run() starts an async program from regular Python code. It creates an event loop, runs the top-level coroutine, and cleans up afterward.

asyncio.run(main())

This is the standard entry point for scripts and command-line programs. In notebooks, web frameworks, and async test runners, an event loop may already be running. In those environments, you normally await the coroutine directly or use the framework's async hooks instead of calling asyncio.run() again

## Async HTTP Calls with httpx

The popular requests library is synchronous. If you call it inside async code, it blocks the event loop. For async HTTP, use a library such as httpx or aiohttp

In [ ]:
%pip install httpx
import asyncio
import httpx
import os

async def call_openai(client: httpx.AsyncClient, prompt: str) -> str:
    """Send a prompt to OpenAI and return the plain text response."""
    response = await client.post(
        "https://api.openai.com/v1/responses",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        json={
            "model": "gpt-5.4-mini",
            "input": prompt,
            "max_output_tokens": 200,
        },
        timeout=30.0,
    )
    response.raise_for_status()
    body = response.json()
    return "".join(
        part["text"]
        for item in body.get("output", [])
        for part in item.get("content", [])
        if part.get("type") == "output_text"
    )

async def main():
    async with httpx.AsyncClient() as client:
        result = await call_openai(client, "Explain async in one sentence.")
        print(result)

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

## asyncio.gather() for Concurrent Tasks

asyncio.gather() takes several awaitables, runs them concurrently, and returns their results in the same order you passed them in

In [16]:
import asyncio
import time

async def call_model(model_name: str, prompt: str) -> dict:
    """Simulate model APIs with different response times."""
    latencies = {
        "large-model": 0.3,
        "balanced-model": 0.4,
        "fast-model": 0.2,
    }
    delay = latencies.get(model_name, 2)

    await asyncio.sleep(delay)
    return {
        "model": model_name,
        "response": f"Answer from {model_name}",
        "latency": delay,
    }

async def main():
    prompt = "Explain distributed systems in 2 sentences."

    start = time.perf_counter()

    results = await asyncio.gather(
        call_model("large-model", prompt),
        call_model("balanced-model", prompt),
        call_model("fast-model", prompt),
    )

    elapsed = time.perf_counter() - start
    print(f"Total time: {elapsed:.1f} seconds")  # ~0.4 seconds, not 0.9

    for r in results:
        print(f"{r['model']}: {r['response']}")

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

Total time: 0.4 seconds
large-model: Answer from large-model
balanced-model: Answer from balanced-model
fast-model: Answer from fast-model


## TaskGroup for Structured Concurrency

asyncio.gather() is still common, especially when you want ordered results. In Python 3.11 and newer, asyncio.TaskGroup is often a better fit when several tasks belong to one operation.

A task group gives child tasks a clear lifetime. Tasks start inside the async with block, and the block does not finish until the tasks finish. If one task fails, the task group cancels the remaining tasks and raises an exception group

In [17]:
import asyncio

async def score_document(document_id: str) -> dict:
    await asyncio.sleep(0.1)
    return {"document_id": document_id, "score": 0.82}

async def main():
    async with asyncio.TaskGroup() as group:
        tasks = [
            group.create_task(score_document(document_id))
            for document_id in ["doc-1", "doc-2", "doc-3"]
        ]

    results = [task.result() for task in tasks]
    print(results)

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

[{'document_id': 'doc-1', 'score': 0.82}, {'document_id': 'doc-2', 'score': 0.82}, {'document_id': 'doc-3', 'score': 0.82}]


Use gather(return_exceptions=True) when partial failure is acceptable and you want to inspect every result. Use TaskGroup when the tasks belong to the same unit of work and one failure should stop the rest

### Error Handling with return_exceptions

By default, if any coroutine inside gather() raises an exception, that exception is raised to the caller. You do not get a normal list of results.

That is sometimes correct. But if you are calling several providers or scoring many independent documents, one timeout may not make the whole batch unusable.

return_exceptions=True changes the behavior. Exceptions are returned as items in the result list, next to successful results

In [22]:
import asyncio

async def flaky_model(name: str) -> str:
    if name == "unstable-model":
        raise TimeoutError(f"{name} timed out")
    await asyncio.sleep(0.3)
    return f"Response from {name}"

async def main():
    results = await asyncio.gather(
        flaky_model("primary-model"),
        flaky_model("unstable-model"),
        flaky_model("backup-model"),
        return_exceptions=True,
    )
    
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"Task {i} failed: {result}")
        else:
            print(f"Task {i}: {result}")

await main()
# asyncio.run(main())

Task 0: Response from primary-model
Task 1 failed: unstable-model timed out
Task 2: Response from backup-model


## Timeouts and Cancellation

Async programs need clear stop conditions. Without timeouts, one slow dependency can keep a request, job, or worker alive far longer than intended.

For individual operations, asyncio.timeout() is a simple built-in guard in Python 3.11 and newer

In [25]:
import asyncio

async def slow_provider_call() -> str:
    await asyncio.sleep(2)
    return "model response"

async def main():
    try:
        async with asyncio.timeout(0.5):
            result = await slow_provider_call()
            print(result)
    except TimeoutError:
        print("Provider call took too long")

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

Provider call took too long


Cancellation is the other side of the same idea. If a user closes the page, a request times out, or a parent task fails, your coroutine may be cancelled.

Most of the time, you should let cancellation propagate. Use try and finally only for cleanup

In [ ]:
import asyncio

async def stream_to_user() -> None:
    try:
        while True:
            await asyncio.sleep(0.2)
            print("sending chunk")
    finally:
        print("cleaning up stream")

async def main():
    task = asyncio.create_task(stream_to_user())
    await asyncio.sleep(0.7)

    task.cancel()

    try:
        await task
    except asyncio.CancelledError:
        print("stream cancelled")

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

sending chunk
sending chunk
sending chunk
cleaning up stream
stream cancelled


Do not catch asyncio.CancelledError unless you plan to re-raise it after cleanup. In modern Python, CancelledError inherits from BaseException, so a normal except Exception block will not catch it. That is intentional: cancellation should usually stop the coroutine.

## Semaphores for Concurrency Limits

A semaphore limits concurrency

A semaphore has a counter. Each task acquires one slot before it starts the protected work. When all slots are taken, the next task waits until a slot is released.

In [29]:
import asyncio
import time

async def call_api(semaphore: asyncio.Semaphore, task_id: int, delay: float) -> str:
    async with semaphore:
        print(f"[{time.perf_counter():.1f}] Task {task_id} started")
        await asyncio.sleep(delay)
        print(f"[{time.perf_counter():.1f}] Task {task_id} done")
        return f"Result {task_id}"

async def main():
    semaphore = asyncio.Semaphore(3)

    tasks = [call_api(semaphore, i, 0.1*(i+1)) for i in range(10)]
    results = await asyncio.gather(*tasks)
    print(f"Got {len(results)} results")
await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

[48191.7] Task 0 started
[48191.7] Task 1 started
[48191.7] Task 2 started
[48191.8] Task 0 done
[48191.8] Task 3 started
[48191.9] Task 1 done
[48191.9] Task 4 started
[48192.0] Task 2 done
[48192.0] Task 5 started
[48192.2] Task 3 done
[48192.2] Task 6 started
[48192.4] Task 4 done
[48192.4] Task 7 started
[48192.6] Task 5 done
[48192.6] Task 8 started
[48193.0] Task 6 done
[48193.0] Task 9 started
[48193.3] Task 7 done
[48193.5] Task 8 done
[48194.0] Task 9 done
Got 10 results


Here is a small client wrapper that applies the same idea to LLM API calls

In [ ]:
import asyncio
import httpx
import os

class RateLimitedClient:
    def __init__(self, max_concurrent: int = 5, timeout: float = 30.0):
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.client = httpx.AsyncClient(timeout=timeout)

    async def __aenter__(self) -> "RateLimitedClient":
        return self

    async def __aexit__(self, exc_type, exc, tb) -> None:
        await self.client.aclose()

    async def call(self, prompt: str, model: str = "gpt-5.4-mini") -> str:
        async with self.semaphore:
            response = await self.client.post(
                "https://api.openai.com/v1/responses",
                headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
                json={
                    "model": model,
                    "input": prompt,
                    "max_output_tokens": 300,
                },
            )
            response.raise_for_status()
            body = response.json()
            return "".join(
                part["text"]
                for item in body.get("output", [])
                for part in item.get("content", [])
                if part.get("type") == "output_text"
            )

async def main():
    prompts = [
        "Summarize this support ticket.",
        "Extract the action items from this meeting note.",
        "Classify this user feedback.",
    ]

    async with RateLimitedClient(max_concurrent=2) as llm:
        results = await asyncio.gather(
            *(llm.call(prompt) for prompt in prompts)
        )

    for result in results:
        print(result)

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

## asyncio.as_completed() for Early Results

asyncio.gather() waits for all tasks to finish, then returns all results at once. Sometimes that is not ideal.

If tasks have different latencies, you may want to process each result as soon as it is ready. asyncio.as_completed() gives you results in completion order, not submission order

In [33]:
import asyncio
import time

async def call_model(name: str, delay: float) -> dict:
    await asyncio.sleep(delay)
    return {"model": name, "delay": delay}

async def main():
    tasks = [
        call_model("fast-model", 0.1),
        call_model("large-model", 0.2),
        call_model("backup-model", 0.1),
        call_model("balanced-model", 0.3),
    ]

    start = time.perf_counter()

    for coro in asyncio.as_completed(tasks):
        result = await coro
        elapsed = time.perf_counter() - start
        print(f"[{elapsed:.1f}s] Got result from {result['model']}")

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

[0.1s] Got result from fast-model
[0.1s] Got result from backup-model
[0.2s] Got result from large-model
[0.3s] Got result from balanced-model


## Async Generators for Streaming LLM Responses

Many LLM APIs support streaming. Instead of waiting for the full response, the application receives small chunks as the model produces output.

In [ ]:
import asyncio
import httpx
import json
import os

async def stream_completion(
    client: httpx.AsyncClient,
    prompt: str,
    model: str = "gpt-5.4-mini",
) -> None:
    """Stream text deltas from an LLM and print them as they arrive."""
    async with client.stream(
        "POST",
        "https://api.openai.com/v1/responses",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        json={
            "model": model,
            "input": prompt,
            "stream": True,
        },
        timeout=60.0,
    ) as response:
        response.raise_for_status()

        async for line in response.aiter_lines():
            if line.startswith("data: ") and line != "data: [DONE]":
                chunk = json.loads(line[6:])
                if chunk.get("type") == "response.output_text.delta":
                    print(chunk["delta"], end="", flush=True)

    print()

async def main():
    async with httpx.AsyncClient() as client:
        await stream_completion(client, "Write a haiku about Python.")

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

ou can also write your own async generators. An async generator is an async def function that uses yield. The caller consumes it with async for.

In [46]:
import asyncio

async def fake_token_stream(text: str, tokens_per_second: int = 10):
    words = text.split()
    delay = 1.0 / tokens_per_second
    for word in words:
        await asyncio.sleep(delay)
        yield word + " "

async def main():
    async for token in fake_token_stream("Async generators work well for streams"):
        print(token, end="", flush=True)
    print()

await main()  # This line is for running in an interactive environment like Jupyter Notebook or IPython
# asyncio.run(main())

Async generators work well for streams 
